# EM619 - Machine Learning
## Assignment 2: Logistic Regression
**Total Marks: 20** | **Questions: 5 (4 marks each)**

### Instructions
- Fill in every `# TODO` block. Do **not** delete function signatures.
- Every question has a mandatory **Observation** markdown cell which you need to answer in 2 to 4 crisp bullet points.
- Set `random_state=42` wherever the function signature exposes it, for reproducible grading.
- Run **Runtime -> Restart session and run all** before submitting; make sure your notebook is free from errors before submission.
- Rename this file to `<name>_EM619_A2.ipynb` before submitting.

| Q | Topic | Marks |
|:-:|:------|:-:|
| 1 | Sigmoid, odds & cross-entropy loss | 4 |
| 2 | Logistic regression from scratch - gradient descent | 4 |
| 3 | Decision boundary & feature transformation | 4 |
| 4 | Multi-class - softmax vs one-vs-rest | 4 |
| 5 | Class imbalance - metrics & threshold tuning | 4 |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_iris, make_circles, make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


---
## Q1. Sigmoid & Cross-Entropy Loss (4 marks)

Logistic regression maps the linear score $z = X\theta$ to a probability with the sigmoid
$\sigma(z) = 1/(1 + e^{-z})$, and is trained by minimising the cross-entropy cost

$$J(\theta) = -\frac{1}{m}\sum_{i=1}^{m}\Big[y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\Big]$$

Implement both, plot them, and verify the log-odds identity $\log\frac{p}{1-p} = z$.

**Marking split:** two functions (2) + plots (1) + observation (1).


In [ ]:
def sigmoid(z):
    '''
    Logistic / sigmoid function:  sigma(z) = 1 / (1 + exp(-z))
    Must work element-wise on a numpy array.
    '''
    # TODO: implement the sigmoid
    return None


def binary_cross_entropy(y, y_hat, eps=1e-12):
    '''
    Mean binary cross-entropy loss:
        J = -(1/m) * sum( y*log(y_hat) + (1-y)*log(1-y_hat) )
    '''
    y_hat = np.clip(y_hat, eps, 1 - eps)   # given: keeps log() finite at 0 and 1

    # TODO: return the mean cross-entropy loss over all samples
    return None


# --- Quick checks (do not modify) ---
z_check = np.array([-10.0, -2.0, 0.0, 2.0, 10.0])
print("sigmoid(z):", np.round(sigmoid(z_check), 4))
print("BCE(y=1, y_hat=0.9):", round(float(binary_cross_entropy(np.array([1.0]), np.array([0.9]))), 4))
print("BCE(y=1, y_hat=0.1):", round(float(binary_cross_entropy(np.array([1.0]), np.array([0.1]))), 4))

# TODO (Plot A): plot the sigmoid over a range of z (say -10 to 10), marking the 0.5
# level. Label the axes and add a title.

# TODO (Plot B): plot the single-example cost against y_hat, once for a true label of
# y = 1 and once for y = 0, on the same axes. Label both curves and add a legend.

# TODO: show that the log-odds of p = sigmoid(z_check) recover z_check.


### Your Observations (Q1)
- What is `sigmoid(0)`? Using that, state where the decision boundary lies in terms of $X\theta$.
- From Plot B: what happens to the cost when $y = 1$ but the model predicts $\hat{y}$ close to 0? Why is that desirable?
- Why is $J(\theta) = \sum_i (y_i - \sigma(x_i\theta))^2$ rejected as the cost function for logistic regression?


---
## Q2. Logistic Regression from Scratch (4 marks)

Dataset: `load_breast_cancer()`. Know more about this dataset at [Link](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)

Logistic regression has no closed-form solution, so parameters are found iteratively. The
gradient of the cross-entropy cost is

$$\frac{\partial J(\theta)}{\partial \theta} = \frac{1}{m} X^{\top}\big(\sigma(X\theta) - y\big)$$

Implement batch gradient descent with this update, then compare against `sklearn`.

**Marking split:** GD + predict functions (2) + loss-curve plot (1) + observation (1).


In [ ]:
def logistic_gd(X, y, alpha=0.1, n_iters=500):
    '''
    Batch gradient descent for logistic regression.
    X already includes a bias column of ones, shape (m, n+1); y is 0/1.

    Returns
    -------
    theta        : np.ndarray, shape (n+1,)
    loss_history : list[float] - cross-entropy loss after each update
    '''
    m, n = X.shape
    theta = np.zeros(n)
    loss_history = []

    for _ in range(n_iters):
        # TODO: one gradient descent step - predict with the current theta, form the
        # gradient of the cross-entropy cost (formula in the question), take a step of
        # size alpha, and record the loss at this iteration.
        pass

    return theta, loss_history


def predict_labels(X, theta, threshold=0.5):
    '''Return 0/1 predictions for design matrix X (with bias column) and theta.'''
    # TODO: return 0/1 labels - class 1 where the predicted probability meets the threshold
    return None


# --- Data setup (do not modify) ---
cancer = load_breast_cancer()
Xb, yb = cancer.data, cancer.target
Xb = StandardScaler().fit_transform(Xb)          # GD needs scaled features
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    Xb, yb, test_size=0.3, random_state=RANDOM_STATE, stratify=yb
)
Xb_train_bias = np.hstack([np.ones((Xb_train.shape[0], 1)), Xb_train])
Xb_test_bias = np.hstack([np.ones((Xb_test.shape[0], 1)), Xb_test])

theta, loss_history = logistic_gd(Xb_train_bias, yb_train, alpha=0.1, n_iters=500)

# TODO: plot loss_history against the iteration number. Label the axes and add a title.

# TODO: print the train and test accuracy of your model.

# TODO: fit sklearn's LogisticRegression on the same training data (raise max_iter if it
# warns about convergence), print its test accuracy, and report the largest absolute
# difference between its coefficients and yours.


### Your Observations (Q2)
- Describe the shape of your loss curve. What does it tell you about the cross-entropy cost surface?
- How does your test accuracy compare with `sklearn`'s? The coefficients are not identical - give one reason why.
- Linear regression has a closed-form normal equation. Why is there no equivalent for logistic regression?


---
## Q3. Decision Boundary & Feature Transformation (4 marks)

Dataset: `make_circles()` - two concentric rings, so no straight line can separate them.

Logistic regression draws a linear boundary ($X\theta = 0$) in whatever feature space it is
given. Fit it on the raw features, then on $\phi(x) = [x_1, x_2, x_1^2, x_2^2]$, and compare.
Also check what the L2 penalty ($J(\theta) = J_1(\theta) + \lambda\theta^{\top}\theta$, where
sklearn's `C` $= 1/\lambda$) does to the coefficients.

**Marking split:** transformation + fitting (2) + decision-boundary plots (1) + observation (1).


In [ ]:
def add_squared_features(X):
    '''
    Given X with columns [x1, x2] (shape (m, 2)), return the transformed
    design matrix [x1, x2, x1^2, x2^2] of shape (m, 4).
    '''
    # TODO: build and return the transformed design matrix
    return None


# --- Data setup and plotting helper (do not modify) ---
Xc, yc = make_circles(n_samples=400, noise=0.08, factor=0.4, random_state=RANDOM_STATE)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.3, random_state=RANDOM_STATE, stratify=yc
)


def plot_decision_boundary(model, X, y, title, transform=None):
    '''Shade the region the model predicts as class 1, then scatter the data on top.'''
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]
    if transform is not None:
        grid = transform(grid)
    Z = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(5.5, 5))
    plt.contourf(xx, yy, Z, alpha=0.25, cmap='coolwarm')
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=25)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title(title)
    plt.show()


# TODO (a): fit a logistic regression on the raw (x1, x2) training data as model_raw,
# print its test accuracy, and plot its decision boundary with the helper above.

# TODO (b): transform train and test with add_squared_features, fit model_poly on the
# transformed data, print its test accuracy, and plot its boundary too (the helper takes
# a transform argument for this).

# TODO (c): refit on the transformed features with a strong and a weak L2 penalty
# (C = 0.01 and C = 100). For each, print the test accuracy and the fitted coefficients.


### Your Observations (Q3)
- Report both test accuracies. Why does the raw-feature model fail - is this a bias or a variance problem?
- Describe the boundary in the second plot. Why is the model still a linear classifier?
- Compare the coefficients for `C=0.01` and `C=100`. What does shrinking `C` do to them, and what is that meant to prevent?


---
## Q4. Multi-Class - Softmax vs One-vs-Rest (4 marks)

Dataset: `load_iris()`. Know more about this dataset at [Link](https://archive.ics.uci.edu/dataset/53/iris)

For $K$ classes the sigmoid generalises to the softmax

$$F(z_k) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}, \qquad \sum_k F(z_k) = 1$$

which reduces to the sigmoid when $K = 2$. Implement softmax, verify that reduction
numerically, and compare multinomial logistic regression against one-vs-rest.

**Marking split:** `softmax` + both fits (2) + confusion-matrix plot (1) + observation (1).


In [ ]:
def softmax(z):
    '''
    Softmax for a 1-D score vector z of length K:
        F(z_k) = exp(z_k) / sum_j exp(z_j)
    Subtracting z.max() before exponentiating is numerically safer and does
    NOT change the result - try to include it.
    '''
    # TODO: implement and return the probability vector
    return None


# --- Quick checks (do not modify) ---
scores = np.array([2.0, 1.0, 0.1])
probs = softmax(scores)
print("softmax(scores):", np.round(probs, 4), "| sums to:", round(float(np.sum(probs)), 6))

# For K = 2, softmax must reduce to the sigmoid:
z_test = 1.3
print("softmax([0, z])[1] =", round(float(softmax(np.array([0.0, z_test]))[1]), 6))
print("sigmoid(z)         =", round(float(sigmoid(z_test)), 6))

# --- Data setup (do not modify) ---
iris = load_iris()
Xi, yi = iris.data, iris.target
Xi_train, Xi_test, yi_train, yi_test = train_test_split(
    Xi, yi, test_size=0.3, random_state=RANDOM_STATE, stratify=yi
)

# TODO: fit a logistic regression as model_softmax - sklearn already uses softmax for
# multi-class (raise max_iter if it warns about convergence). Print its test accuracy and
# the shape of its coefficient matrix.

# TODO: fit a one-vs-rest logistic regression as model_ovr and print its test accuracy.

# TODO: print the predicted probability vector of the first test sample and check it sums to 1.

# TODO (Plot): plot the confusion matrix of model_softmax on the test set, with the class
# names as labels and a title.


### Your Observations (Q4)
- State the two properties softmax guarantees, and confirm both from your printed values.
- What is `model.coef_.shape`, and what does each row represent? How does one-vs-rest differ, and how did the two accuracies compare?
- Which classes does the confusion matrix show being mixed up? Does that match what you know about Iris?


---
## Q5. Class Imbalance - Metrics & Threshold Tuning (4 marks)

Dataset: synthetic imbalanced data via `make_classification` (~5% positives).

Accuracy is misleading when one class dominates. Train a plain model and a class-weighted one,
then tune the decision threshold $\tau$, judging both on precision / recall / F1 / ROC-AUC
instead of accuracy.

**Marking split:** both helper functions + the two fits (2) + metrics-vs-threshold plot (1) + observation (1).


In [ ]:
def evaluate(y_true, y_pred, label):
    '''Print accuracy, precision, recall and F1 for a set of 0/1 predictions.'''
    # TODO: print the label followed by accuracy, precision, recall and F1 on one line.
    # (Pass zero_division=0 so a model that predicts no positives does not crash.)
    pass


def metrics_vs_threshold(model, X_test, y_test, thresholds):
    '''
    For each tau in thresholds, predict class 1 when P(y=1|x) >= tau and
    record precision, recall and F1.

    Returns three lists: precisions, recalls, f1s
    '''
    probs = model.predict_proba(X_test)[:, 1]     # given
    precisions, recalls, f1s = [], [], []

    for tau in thresholds:
        # TODO: turn the probabilities into 0/1 predictions at this threshold and
        # append the resulting precision, recall and F1 to the three lists.
        pass

    return precisions, recalls, f1s


# --- Data setup (do not modify) ---
X_imb, y_imb = make_classification(
    n_samples=4000, n_features=8, n_informative=4, n_redundant=0,
    weights=[0.95, 0.05], flip_y=0.0, class_sep=2.5, random_state=RANDOM_STATE
)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=RANDOM_STATE, stratify=y_imb
)
print("Train class counts:", np.bincount(ym_train),
      f"| positives: {100 * ym_train.mean():.2f}%")

# TODO (a): fit a plain logistic regression as model_plain, call evaluate on its test
# predictions with the label "plain", and print its confusion matrix.

# TODO (b): same again as model_bal, but with sklearn's balanced class weights
# (class_weight='balanced'), labelled "class_weight=balanced".

# TODO (c): sweep the decision threshold from 0.05 to 0.95 with metrics_vs_threshold on the
# plain model, plot precision, recall and F1 against the threshold on the same axes, and
# print the threshold that gives the best F1.

# TODO: print the ROC-AUC of both models.


### Your Observations (Q5)
- Compare the plain model's accuracy with its recall. Using the confusion matrix, explain why accuracy is misleading here.
- What does `class_weight='balanced'` change in the cost function? Which metric improved, and which one got worse?
- Report your best-F1 threshold. Is it above or below 0.5, and why does that direction make sense?
- Both models have nearly the same ROC-AUC despite very different predictions. What does that tell you?


---
### Submission Checklist
- [ ] All `# TODO` blocks completed, no `None` / `pass` placeholders remain
- [ ] All plots render without errors
- [ ] All 5 Observation cells filled in
- [ ] Ran **Restart and run all** with no errors
- [ ] File renamed to `<name>_EM619_A2.ipynb`
